# RetailPulse — Day 3 EDA and Business Insights

This notebook analyzes only the Day 2 outputs in `data/processed/`. It does not read or modify `data/raw/`. It calculates monthly revenue, product/category performance, customer purchase frequency, catalog margins, and inventory reorder risk. Run from the repository root (or `notebooks/`).

The companion report contains findings calculated from the repository CSVs. This notebook is committed without execution outputs; run it to reproduce tables and charts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Run from the repository root or from the notebooks/ directory.
cwd = Path.cwd()
ROOT = cwd if (cwd / "data" / "processed").exists() else cwd.parent
DATA = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(DATA / "sales_clean.csv", parse_dates=["date"])
customers = pd.read_csv(DATA / "customers_clean.csv", parse_dates=["registration_date"])
products = pd.read_csv(DATA / "products_clean.csv")
inventory = pd.read_csv(DATA / "inventory_clean.csv", parse_dates=["date"])

print({name: frame.shape for name, frame in {
    "sales": sales, "customers": customers, "products": products, "inventory": inventory
}.items()})

In [ ]:
# Verify the Day 2 columns expected below before calculating metrics.
expected = {
    "sales": {"transaction_id", "customer_id", "product_id", "date", "quantity", "unit_price", "total_amount"},
    "customers": {"customer_id", "registration_date"},
    "products": {"product_id", "product_name", "category", "unit_cost", "selling_price", "unit_margin", "margin_pct"},
    "inventory": {"product_id", "date", "stock_quantity", "reorder_level", "lead_time", "stock_gap", "below_reorder"},
}
frames = {"sales": sales, "customers": customers, "products": products, "inventory": inventory}
for name, required in expected.items():
    missing = required - set(frames[name].columns)
    if missing:
        raise ValueError(f"{name} is missing expected columns: {sorted(missing)}")

print("Rows:", {name: len(frame) for name, frame in frames.items()})
print("Sales date range:", sales["date"].min().date(), "to", sales["date"].max().date())
print("Missing cells:", {name: int(frame.isna().sum().sum()) for name, frame in frames.items()})

In [ ]:
# Monthly recorded net revenue
monthly = sales.assign(month=sales["date"].dt.to_period("M").dt.to_timestamp()).groupby("month", as_index=False)["total_amount"].sum()
display(monthly)
ax = monthly.plot(x="month", y="total_amount", kind="line", marker="o", figsize=(11, 4), legend=False)
ax.set(title="Monthly net revenue", xlabel="Month", ylabel="Revenue")
ax.grid(alpha=.25)
plt.tight_layout()
plt.savefig(FIGURES / "monthly_revenue.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Product and category performance by recorded net revenue
sales_products = sales.merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
category_sales = sales_products.groupby("category", as_index=False).agg(revenue=("total_amount", "sum"), units=("quantity", "sum")).sort_values("revenue", ascending=False)
product_sales = sales_products.groupby(["product_id", "product_name", "category"], as_index=False).agg(revenue=("total_amount", "sum"), units=("quantity", "sum")).sort_values("revenue", ascending=False)
display(category_sales)
display(product_sales.head(10))
ax = category_sales.sort_values("revenue").plot.barh(x="category", y="revenue", figsize=(9, 5), legend=False)
ax.set(title="Net revenue by category", xlabel="Revenue", ylabel="")
plt.tight_layout()
plt.savefig(FIGURES / "category_revenue.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Customer purchase frequency and spend
customer_sales = sales.groupby("customer_id", as_index=False).agg(transactions=("transaction_id", "nunique"), revenue=("total_amount", "sum"), units=("quantity", "sum"))
customer_summary = customer_sales["transactions"].describe()
repeat_customers = int((customer_sales["transactions"] > 1).sum())
print("Customers represented:", len(customer_sales))
print("Repeat customers:", repeat_customers, f"({repeat_customers / len(customer_sales):.1%})")
print("Mean transactions per customer:", customer_sales["transactions"].mean())
print("Mean transaction value:", sales["total_amount"].mean())
display(customer_summary)
display(customer_sales.sort_values("revenue", ascending=False).head(10))

In [ ]:
# Product catalog unit economics; these are not realized transaction margins.
catalog_margin = products.groupby("category", as_index=False).agg(
    mean_margin_pct=("margin_pct", "mean"),
    median_margin_pct=("margin_pct", "median"),
    mean_unit_margin=("unit_margin", "mean"),
    products=("product_id", "nunique"),
).sort_values("mean_margin_pct", ascending=False)
print("Overall mean catalog margin %:", products["margin_pct"].mean())
print("Overall median catalog margin %:", products["margin_pct"].median())
display(catalog_margin)
ax = catalog_margin.sort_values("mean_margin_pct").plot.barh(x="category", y="mean_margin_pct", figsize=(9, 5), legend=False)
ax.set(title="Mean catalog margin by category", xlabel="Mean margin (%)", ylabel="")
plt.tight_layout()
plt.savefig(FIGURES / "catalog_margin_by_category.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Inventory reorder screen from the processed snapshot
inventory_risk = inventory.merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
inventory_risk["below_reorder"] = inventory_risk["below_reorder"].astype(str).str.lower().eq("true")
risk_summary = inventory_risk.groupby("category", as_index=False).agg(
    observations=("product_id", "nunique"),
    below_reorder=("below_reorder", "sum"),
    mean_lead_time_days=("lead_time", "mean"),
).sort_values("below_reorder", ascending=False)
risk_items = inventory_risk.loc[inventory_risk["below_reorder"]].sort_values("stock_gap")
print(f"Below reorder: {len(risk_items)} / {len(inventory_risk)} ({len(risk_items) / len(inventory_risk):.1%})")
print("Mean lead time:", inventory_risk["lead_time"].mean(), "days")
display(risk_summary)
display(risk_items[["product_id", "product_name", "category", "date", "stock_quantity", "reorder_level", "stock_gap", "lead_time"]])
ax = risk_summary.sort_values("below_reorder").plot.barh(x="category", y="below_reorder", figsize=(9, 5), legend=False)
ax.set(title="Below-reorder observations by category", xlabel="Products below reorder level", ylabel="")
plt.tight_layout()
plt.savefig(FIGURES / "reorder_risk_by_category.png", dpi=160, bbox_inches="tight")
plt.show()